# today is a very important day marking the launch day.

## THE IDEAS, STEP BY STEP

### The front door (an allow-list)

Think of a site gate. A guard with a list of banned people (a deny-list) lets in everyone he has not heard of. A guard with a list of two allowed names (an allow-list) lets in nobody else. Your API has routes only you should use (`/metrics`, `/readyz`, `/statusz`, `/docs`). With a deny-list, the first route you forget is open to the world.

So the door, a small program called Caddy, allows exactly two things:

`POST` `/ask` goes to the API. The API still checks the API key.
`GET` `/healthz` goes to the API (`HEAD` works too, for uptime services).

Everything else gets “`404 not found`” from the door itself and never reaches the API. **404** says nothing, while **401** would tell a stranger the route exists.

The match is exact and uses the decoded path. People try `/Metrics`, `//metrics`, `/%6Detrics` and `/ask/../metrics`. The door decodes these first and then compares to the two allowed paths, so none of them pass. The tests send 52 such spellings, with `GET` and `POST`, and check at the API itself that none arrived.

## The door also:

- refuses a question body over 20 KB;
- ends connections that drip bytes slowly;
- hides which software it is;
- gives each request one id that follows it into the API’s log.

It writes one log line per request with no headers, because the API key travels in a header and must never reach a log.

Backups that count

A backup you have never restored is a hope. Cell 6 makes a backup, checks the files, and then practises a restore into throwaway copies of your data, so your real data is never touched.

Why not just copy the SQLite file? SQLite in WAL mode keeps the newest rows in a side file (the -wal file). Copying only the main file silently loses them. The tool uses SQLite’s own online backup, which reads everything consistently while the app keeps writing. I tested this on purpose: a test makes a writer that never checkpoints, and a plain file copy is caught as wrong.

Redis keys with a time limit (rate limits) run out by themselves. A snapshot taken at 13:00 and loaded at 13:05 holds fewer keys. An earlier version of my check called that a failed restore. Now it requires every key that never expires to come back, and accepts the time-limited ones being gone.

## A real restore (only if you need it) does this in order:

- stops the API, so nothing writes;
- saves what is there now as before_restore_...;
- stops everything;
- replaces the data;
- checks every restored file by checksum.

It refuses to empty a volume to put nothing back, and refuses a backup made by a different project.

Alerts, and who watches the watcher

### A separate container, the watchdog, looks every 30 seconds at the API, the door and the backups. It follows these rules:

It tells you only when something changes (FIRING, then RESOLVED), once, with several problems in one message.
It waits for two bad looks in a row, so one blip is not an outage.
It stays quiet for 15 minutes after a start, while the models load.
A critical problem that stays is repeated every 6 hours. A warning is not.
Every round it pings a dead-man’s switch (an outside web service). If your whole computer dies, the pings stop, and that service tells you, because the dead computer cannot.

It also tests the door from the inside. If /metrics ever answers, it raises a critical alert at once.

## The launch check (commissioning)

Like handing a building over, someone outside checks it against a list. check_launch.py looks from the outside: door, containers, backups, alerts, secrets, one real question. It then reads the door’s log and fails if one of your keys is in it.

Its last line is honest. “READY TO LAUNCH” appears only if everything was checked. If something was skipped (no public address yet, no Docker, no real question), it says “NOTHING FAILED in what was checked. NOT checked this time: ...”.

## The public address (optional)

A Cloudflare quick tunnel is free and gives a random https address that leads to your door, not to your API. It is good for testing and for a few friends. The address changes every time the tunnel restarts.

In [2]:
# Checking files, packages, Docker

import os, sys, subprocess, time, json, pathlib, re, importlib
sys.path.insert(0, os.getcwd())

# --- 1. The Day 21 files must be in this folder (make_container_files.py replaces the Day 20 one)
FILES = ["make_frontdoor.py", "backup_tools.py", "notifiers.py", "watchdog.py", "check_launch.py", "manage_keys.py", "tunnel_tools.py",
         "fakeservices.py", "doorkit.py", "make_container_files.py",
         "test_frontdoor.py", "test_backup.py", "test_notifiers.py", "test_watchdog.py", "test_launch_files.py", "test_check_launch.py",
         "test_manage_keys.py", "test_tunnel_tools.py", "test_windows_pipes.py",
         "alert_rules.py", "monitoring.py", "check_monitoring.py", "log_report.py", "prepare_monitoring.py", "testkit.py",       # Day 20
         "serve.py", "api_layer.py", "container_check.py", "make_requirements.py", "usage_context.py"]                           # Days 17-19
absent = [f for f in FILES if not os.path.exists(f)]
assert not absent, f"❌ Missing from {os.getcwd()}: {absent}  (extract day21_tools.zip into this folder; choose 'Replace' when asked)"
assert "WATCHDOG_FILES" in pathlib.Path("make_container_files.py").read_text(encoding="utf-8"), \
    "❌ make_container_files.py is still the Day 20 one: extract day21_tools.zip again and choose Replace"
for must in ("rag_pipeline/pipeline.py", ".env", "Dockerfile", "docker-compose.yml", "docker_healthcheck.py"):
    assert os.path.exists(must), f"❌ {must} is missing. It comes from Day 18 (freeze, .env) / Day 19 (Docker files)."
env_text = pathlib.Path(".env").read_text(encoding="utf-8-sig")
assert re.search(r"^RAG_METRICS_KEY=.{16,}", env_text, re.M), "❌ RAG_METRICS_KEY is missing from .env: run Day 20 Cell 4 first (the watchdog needs it)."
assert re.search(r"^RAG_API_KEYS=.+:.+", env_text, re.M), "❌ RAG_API_KEYS is missing from .env (Day 17/18)."

# --- 1b. If this notebook was open on Day 20, Python still remembers the OLD versions of these tools. Forget them.
for name in [f[:-3] for f in FILES if not f.startswith("test")] + ["container_patch", "docker_healthcheck"]:
    sys.modules.pop(name, None)
importlib.invalidate_caches()

# --- 2. Packages the tools need
for mod, pkg in (("httpx", "httpx"), ("dotenv", "python-dotenv"), ("uvicorn", "uvicorn"), ("fastapi", "fastapi"),
                 ("yaml", "pyyaml"), ("prometheus_client", "prometheus-client")):
    try:
        __import__(mod)
    except ImportError:
        raise AssertionError(f"❌ {pkg} is not installed. Run this in a cell by itself, then run this cell again:   %pip install {pkg}")

# --- 3. A helper that runs docker commands and gives back (exit code, text)
def dc(*args, timeout=1800):
    try:
        r = subprocess.run(["docker", *args], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=timeout)
    except FileNotFoundError:
        return 127, "docker was not found on this computer"
    return r.returncode, (r.stdout + r.stderr).strip()

# --- 4. Docker, and the stack from Day 19/20
code, out = dc("compose", "ps", "--format", "{{.Service}} {{.State}}")
if code != 0:
    print("⚠️  Docker is not running or not installed:", out[:200])
    print("   Cells 2-4 still work. Start Docker Desktop (wait until the whale icon stops moving) before the terminal step.")
else:
    print("Docker stack right now:", out.replace("\n", " | ") or "(nothing running; that is fine, the terminal step starts it)")
print(f"✅ ready. Python {sys.version.split()[0]} | folder: {os.getcwd()}")


Docker stack right now: (nothing running; that is fine, the terminal step starts it)
✅ ready. Python 3.11.14 | folder: c:\Users\rodne\Desktop\4 MONTHS JOURNEY\MONTH4-2


In [9]:
# The nine test files; nothing sent to anyone

# Runs the nine Day 21 test files. Fake phones, fake mail servers and fake uptime services run on this computer, so nothing is sent
# to anyone and no tokens are spent. The last one (test_backup.py) really uses Docker: it makes throwaway volumes and deletes them.
# It takes a few minutes. "skipped" means a test could not run on this computer (for example: no `caddy` program): that is not a failure.
SUITES = ["test_windows_pipes.py", "test_tunnel_tools.py", "test_manage_keys.py", "test_launch_files.py", "test_notifiers.py", "test_watchdog.py",
          "test_frontdoor.py", "test_check_launch.py", "test_backup.py"]
failed = []
for s in SUITES:
    t0 = time.time()
    r = subprocess.run([sys.executable, s], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=1800)
    last = [l for l in r.stdout.splitlines() if " passed" in l][-1:] or ["(no result line)"]
    ok = r.returncode == 0
    print(("✅" if ok else "❌"), f"{s:<24}", last[0].strip(), f"  ({time.time() - t0:.0f}s)")
    if not ok:
        failed.append(s)
        print("   ", "\n    ".join(l for l in r.stdout.splitlines() if l.startswith("FAIL") or l.startswith("  File") or "Error" in l)[:1800])
        print("   ", r.stderr[-600:])
assert not failed, f"❌ failing: {failed}. Send me the lines above."
print(f"\n✅ all {len(SUITES)} test files pass on this computer")

✅ test_windows_pipes.py    6/6 passed   (5s)
✅ test_tunnel_tools.py     8/8 passed   (0s)
✅ test_manage_keys.py      8/8 passed   (3s)
✅ test_launch_files.py     16/16 passed   (3s)
✅ test_notifiers.py        18/18 passed (1 skipped)   (27s)
✅ test_watchdog.py         24/24 passed   (13s)
✅ test_frontdoor.py        5/5 passed (14 skipped: no caddy program on this computer)   (2s)
✅ test_check_launch.py     15/31 passed, 16 skipped   (8s)
✅ test_backup.py           23/23 passed   (330s)

✅ all 9 test files pass on this computer


In [10]:
# Door settings, new Docker files, also keeps private files out of Git

# 1. Writes the front door's settings (the Caddyfile). 2. Writes the Docker files again: they now also contain the door (proxy),
# the watchdog and an OFF-by-default tunnel. 3. Keeps private files out of git.
import difflib
from make_frontdoor import write_caddyfile, CLOSED_PATHS
from container_check import check_container_readiness, fix_env_file
from make_requirements import write_requirements
from make_container_files import make_container_files

print("Caddyfile:", write_caddyfile("."))
for line in pathlib.Path("Caddyfile").read_text(encoding="utf-8").splitlines():
    if line.strip().startswith(("@ask", "@health", "max_size", "handle ", "respond ")):
        print("   ", line.strip())
print(f"   (everything except POST /ask and GET /healthz gets 404 from the door itself; the API never sees it. {len(CLOSED_PATHS)} tricky spellings are tested in Cell 7.)")

os.makedirs("backups", exist_ok=True)           # create it ourselves, so Docker does not create it as a locked folder
gi = pathlib.Path(".gitignore")
lines = gi.read_text(encoding="utf-8").splitlines() if gi.exists() else []
add = [x for x in ("backups/", "ntfy_topic.txt", "new_key_*.txt", "alert_test.json", ".env.before_*", "Caddyfile.previous", "*.previous", "run_backup.cmd") if x not in lines]
if add:
    gi.write_text("\n".join(lines + add) + "\n", encoding="utf-8")
    print("added to .gitignore:", ", ".join(add))

for line in fix_env_file(".env"):
    print("🔧", line)
check = check_container_readiness("rag_pipeline", ".")
assert check.ok, check.summary()
reqs = write_requirements("rag_pipeline", ".")
old = {n: pathlib.Path(n).read_text(encoding="utf-8").splitlines() for n in ("Dockerfile", ".dockerignore", "docker-compose.yml")}
written = make_container_files("rag_pipeline", ".", check, reqs)
print("\nwrote:", ", ".join(written))
for n, before in old.items():
    after = pathlib.Path(n).read_text(encoding="utf-8").splitlines()
    changes = [l for l in difflib.unified_diff(before, after, lineterm="", n=0) if l[:1] in "+-" and l[:3] not in ("+++", "---")]
    print(f"{n}: " + ("unchanged" if not changes else f"{len(changes)} changed line(s)"))
    if n != "docker-compose.yml":
        for l in changes:
            print("    ", l)
code, out = dc("compose", "config", "--services")
print("services now:", out.replace("\n", ", ") if code == 0 else "❌ " + out[-800:])
code, out = dc("compose", "config", "-q")
print("docker-compose.yml:", "✅ valid" if code == 0 else "❌ " + out[-800:])
assert code == 0


Caddyfile: .\Caddyfile
    @ask expression `{http.request.method} == "POST" && {http.request.uri.path} == "/ask"`
    @health expression `({http.request.method} == "GET" || {http.request.method} == "HEAD") && {http.request.uri.path} == "/healthz"`
    handle @ask {
    max_size 20KB
    handle @health {
    handle {
    respond "not found" 404
    respond "{http.error.status_text}" {http.error.status_code}
   (everything except POST /ask and GET /healthz gets 404 from the door itself; the API never sees it. 52 tricky spellings are tested in Cell 7.)
added to .gitignore: backups/, ntfy_topic.txt, new_key_*.txt, alert_test.json, .env.before_*, Caddyfile.previous, run_backup.cmd

wrote: Dockerfile, .dockerignore, docker-compose.yml, docker_healthcheck.py
Dockerfile: 4 changed line(s)
     -COPY --chown=app:app serve.py api_layer.py docker_healthcheck.py monitoring.py ./
     -COPY --chown=app:app  ./ ./
     +COPY --chown=app:app serve.py api_layer.py docker_healthcheck.py monitoring.py a

In [11]:
# secret nfty topic for phone alerts

# Makes a secret ntfy "topic" (free; no account). The watchdog sends alerts to that topic and the ntfy app on your phone listens to it.
# The topic is written to .env and to the file ntfy_topic.txt; it is NOT printed here, because the topic name is the password.
import notifiers

added = notifiers.ensure_env_values(".env", {"ALERT_UTC_OFFSET_HOURS": "3"})        # Uganda is UTC+3: alert times then read in your time
print("added to .env:", added or "nothing (already there)")
print(notifiers.ensure_ntfy_topic(".env"))
print()
print("\n".join(notifiers.describe(notifiers.read_env_file(".env"))))
print("""
NOW, ON YOUR PHONE (2 minutes):
  1. Install the free app "ntfy" (Play Store / App Store).
  2. Open it, tap +, "Subscribe to topic".
  3. Open ntfy_topic.txt in your project folder, copy the second line, paste it as the topic name. Leave the server as ntfy.sh. Subscribe.
Then do the terminal step below. A test message will arrive in Cell 5.""")

added to .env: ['ALERT_UTC_OFFSET_HOURS']
NTFY_TOPIC_SECRET created in .env. The topic to type into your phone is in ntfy_topic.txt

–  telegram: not set up
✅ ntfy: set up
–  email: not set up

NOW, ON YOUR PHONE (2 minutes):
  1. Install the free app "ntfy" (Play Store / App Store).
  2. Open it, tap +, "Subscribe to topic".
  3. Open ntfy_topic.txt in your project folder, copy the second line, paste it as the topic name. Leave the server as ntfy.sh. Subscribe.
Then do the terminal step below. A test message will arrive in Cell 5.


**Terminal setup**

docker compose build

docker compose up -d

In [12]:
# wait for when the container is ready, look around, send a test alert from inside the watchdog container

# Run AFTER the terminal step (docker compose build, docker compose up -d).
# Waits until the service is ready, shows the containers, tries the front door, and sends a test alert FROM INSIDE the watchdog container.
import httpx
from check_monitoring import keys_from_env

K = keys_from_env(".env")
BASE = K["base_url"]                                                    # the API directly, on this computer (port 8080)
from dotenv import dotenv_values
DOOR = f"http://127.0.0.1:{dotenv_values('.env').get('RAG_DOOR_PORT') or '8088'}"      # the front door, on this computer (port 8088)

def wait_ready(max_seconds=900):
    """Poll /readyz until it says 200. After a rebuild the models come from the saved volume, so it takes a few minutes."""
    t0, last = time.time(), "nothing yet"
    while time.time() - t0 < max_seconds:
        try:
            r = httpx.get(f"{BASE}/readyz", timeout=5)
            if r.status_code == 200:
                return time.time() - t0
            last = f"readyz {r.status_code}"
        except Exception as e:
            last = type(e).__name__
        time.sleep(3)
    raise AssertionError(f"❌ not ready after {max_seconds}s (last: {last}). Read the log:  docker compose logs --tail 80 rag")

print(f"API ready after {wait_ready():.0f}s")
code, out = dc("compose", "ps", "--format", "table {{.Service}}\t{{.State}}\t{{.Health}}\t{{.Ports}}")
print(out)

print("\n--- the front door:", DOOR)
for method, path in (("GET", "/healthz"), ("GET", "/metrics"), ("GET", "/docs"), ("GET", "/readyz"), ("POST", "/ask")):
    r = httpx.request(method, DOOR + path, json={"question": "x"} if method == "POST" else None, timeout=10)
    note = {"/healthz": "open, on purpose", "/metrics": "closed: the door answers 404 itself", "/docs": "closed", "/readyz": "closed",
            "/ask": "reaches the API, which asks for a key (401)"}[path]
    print(f"  {method:<4} {path:<9} -> HTTP {r.status_code}   ({note})")
assert httpx.get(DOOR + "/healthz").status_code == 200 and httpx.get(DOOR + "/metrics").status_code == 404

print("\n--- the watchdog (reads its settings from the container, never printing a secret)")
code, out = dc("compose", "exec", "-T", "watchdog", "python", "watchdog.py", "--config")
print(out)
print("\n--- a test alert, sent from INSIDE the watchdog container (the same path real alerts take):")
code, out = dc("compose", "exec", "-T", "watchdog", "python", "notifiers.py", "--test", "--from-environment")
print(out)
assert code == 0, "❌ the test alert did not get out: read the ❌ line above (is the topic in .env? does the container have internet?)"
import notifiers
results = {m.group(2): ("ok" if m.group(1) == "✅" and "delivered" in l else "failed")
           for l in out.splitlines() for m in [re.match(r"^(✅|❌) ([a-z]+):", l.strip())] if m}
if results:
    notifiers.write_test_record("alert_test.json", results)           # the launch check (Cell 7) reads this note: "a test alert was accepted by ntfy"
print("\n✅ the door is shut where it should be, and alerts leave the container. LOOK AT YOUR PHONE: 'TEST from rag-api' should be there.")

API ready after 1s
SERVICE    STATE     <no value>   PORTS
proxy      running                127.0.0.1:8088->8080/tcp
rag        running   healthy      127.0.0.1:8080->8000/tcp
redis      running   healthy      6379/tcp
watchdog   running   healthy      8000/tcp

--- the front door: http://127.0.0.1:8088
  GET  /healthz  -> HTTP 200   (open, on purpose)
  GET  /metrics  -> HTTP 404   (closed: the door answers 404 itself)
  GET  /docs     -> HTTP 404   (closed)
  GET  /readyz   -> HTTP 404   (closed)
  POST /ask      -> HTTP 401   (reaches the API, which asks for a key (401))

--- the watchdog (reads its settings from the container, never printing a secret)
label: rag-api
API:   http://rag:8000
door:  http://proxy:8080
public: not watched
backups: /backups (newest must be under 48 h old)
heartbeat: off
every 30s; quiet for the first 900s after start
delivery: ntfy

--- a test alert, sent from INSIDE the watchdog container (the same path real alerts take):
✅ ntfy: delivered

✅ the door i

In [13]:
# backuo, check, practicing a restore

# A backup you have never restored is a hope, not a backup. Three steps: make one, check it, practise restoring it.
# Step 3 restores into THROWAWAY volumes (named ...-drill-...) and deletes them: your real data is never touched.
def tool(*args):
    r = subprocess.run([sys.executable, "backup_tools.py", *args], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=1800)
    print(r.stdout.rstrip() + (("\n" + r.stderr.rstrip()) if r.returncode else ""))
    return r.returncode

print("=== 1. back up (the stack keeps running; the copy is consistent even while it writes)")
assert tool("backup") == 0, "❌ the backup failed: read the ❌ line above"
print("\n=== 2. check the files (no Docker needed: checksums, databases open, row counts)")
assert tool("verify") == 0
print("\n=== 3. practise a restore")
assert tool("drill") == 0, "❌ the practice restore failed: the backup would not have saved you. Send me the lines above"
print("\n=== what you have")
tool("list")
size = sum(f.stat().st_size for f in pathlib.Path("backups").rglob("*") if f.is_file())
print(f"\nbackups folder: {size / 1024**2:.1f} MB. It holds your conversations: keep it private, and never upload it anywhere public.")
print("NOT in it, on purpose: .env (your keys). Keep a copy of .env in your password manager.")


=== 1. back up (the stack keeps running; the copy is consistent even while it writes)
✅ checkpoints/rag_checkpoints.db: SQLite integrity ok, 580 rows in 2 table(s)
✅ redis/dump.rdb: 1,732 bytes, checksum ok
✅ redis: snapshot of 5 key(s)
✅ backup 20261027-202318 saved and verified (1.2 MB, 10s) in c:\Users\rodne\Desktop\4 MONTHS JOURNEY\MONTH4-2\backups\20261027-202318

=== 2. check the files (no Docker needed: checksums, databases open, row counts)
✅ checkpoints/rag_checkpoints.db: SQLite integrity ok, 580 rows in 2 table(s)
✅ redis/dump.rdb: 1,732 bytes, checksum ok
✅ redis: snapshot of 5 key(s)

=== 3. practise a restore
practising on c:\Users\rodne\Desktop\4 MONTHS JOURNEY\MONTH4-2\backups\20261027-202318
✅ checkpoints: files restored with the app user as owner
✅ redis: snapshot loaded and the append-only file written
✅ checkpoints/rag_checkpoints.db: opened inside the app image; integrity ok, 580 rows, as saved
✅ redis: started the way compose starts it (--appendonly yes) and holds

In [19]:
# the launch check

# The launch check: looks at your service from the OUTSIDE (the front door), your backups, your alerts and your secrets.
# --ask also sends ONE real question through the door (a few thousand tokens), then reads the door's log to make sure none of your keys got into it.
# The LAST line is honest about what was looked at: "READY TO LAUNCH" only when everything was checked, public address included (Cell 9).
# Before that it says "NOTHING FAILED in what was checked ... NOT checked this time: the public address". That is correct and expected.
r = subprocess.run([sys.executable, "check_launch.py", "--ask"], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=900)
print(r.stdout.rstrip())
if r.stderr.strip():
    print(r.stderr[-800:])
assert r.returncode == 0, "❌ read the 'NOT READY' list at the bottom: each line says what to fix. Fix it, then run this cell again."


1. THE FRONT DOOR  (http://127.0.0.1:8088)
✅ the front door answers /healthz (HTTP 200)
✅ HEAD /healthz works too, for uptime services (HTTP 200)
✅ 52 spellings of closed routes (/metrics, /readyz, /statusz, /docs ...) all answer 404, for GET and for POST
✅ GET /ask is closed (HTTP 404): only POST gets in
✅ the door does not tell strangers what software it is (no Server header)
✅ a question with no key is refused by the API behind the door (HTTP 401): the door reaches the API and the API checks keys
✅ a question with a wrong key is refused (HTTP 401)
✅ a 30 KB question is refused (HTTP 413; 413 = Payload Too Large). The door lets at most 20 KB of any question through to the API

2. THE PUBLIC ADDRESS
✅ the public address answers /healthz (HTTP 200)
✅ 12 closed routes are closed from the internet too
✅ a question with no key is refused from the internet (HTTP 401)

3. THE CONTAINERS
✅ every published port is limited to this computer (127.0.0.1)
✅ rag: running, healthy
✅ redis: running,

In [15]:
# fire drill, switching the API off and watching my phone

# A fire drill: switch the API OFF on purpose and prove that your phone hears about it, then switch it back on.
# The watchdog checks every 30 seconds and waits for two failed checks in a row, so the alert comes after about a minute.
# On your phone you will see, in this order:  CRITICAL (the API and the door are down, in ONE message)  ->  WARNING process_restarted
# (the API really was restarted: that is true, so it says so)  ->  RESOLVED (back to normal). The same happens whenever YOU restart
# the API on purpose (new key, new settings): the watchdog cannot tell a planned restart from a crash, and a few minutes of silence are not normal.
assert "DOOR" in globals(), "❌ run Cell 5 first: it defines the helpers this cell uses (and it restarts nothing)"
def watchdog_log(since=None):
    args = ["compose", "logs", "--no-color"] + (["--since", since] if since else []) + ["watchdog"]
    return dc(*args)[1]

# 0. The watchdog stays quiet until it has seen the API healthy once (so it does not shout while the models are still loading).
#    It writes one line when that happens. Wait for it, otherwise the drill would test the quiet period instead of the alert.
print("0. waiting until the watchdog has seen the API healthy at least once ...")
t0 = time.time()
while "answers /metrics" not in watchdog_log() and time.time() - t0 < 300:
    time.sleep(5)
assert "answers /metrics" in watchdog_log(), "❌ the watchdog never managed to read /metrics: docker compose logs --tail 40 watchdog  (is RAG_METRICS_KEY in .env the one the API uses?)"
mark = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())          # only log lines written AFTER this moment count

print("1. stopping the API container (rag) ...")
code, out = dc("compose", "stop", "rag")
assert code == 0, out
t0, seen = time.time(), False
while time.time() - t0 < 240:
    time.sleep(5)
    if "FIRING api_unreachable" in watchdog_log(mark):
        seen = True
        break
if not seen:
    print("last lines of the watchdog's log:")
    print("\n".join(l[:240] for l in watchdog_log(mark).splitlines()[-12:]))
    dc("compose", "start", "rag")
    raise AssertionError("❌ the watchdog did not announce the stopped API within 4 minutes (the API was started again). Send me the lines above.")
print(f"2. {time.time() - t0:.0f}s after stopping it, the watchdog announced it. Lines from its log:")
for l in [l for l in watchdog_log(mark).splitlines() if "FIRING" in l][-3:]:
    print("    ", re.sub(r"^.*?(FIRING)", r"\1", l)[:200])
r = httpx.get(DOOR + "/healthz", timeout=10)
print(f"3. the door itself still answers, honestly: /healthz -> HTTP {r.status_code} (502 = the API behind it is down)")
print("   👉 LOOK AT YOUR PHONE: ONE CRITICAL message should have arrived (it names api_unreachable and door_down together).")

print("\n4. starting the API again ...")
code, out = dc("compose", "start", "rag")
assert code == 0, out
print(f"   ready after {wait_ready():.0f}s")
t1, resolved = time.time(), False
while time.time() - t1 < 180:
    if "RESOLVED api_unreachable" in watchdog_log(mark):
        resolved = True
        break
    time.sleep(5)
assert resolved, "❌ the watchdog never said RESOLVED: docker compose logs --tail 60 watchdog"
print("5. the watchdog announced that it is back.   👉 LOOK AT YOUR PHONE: a RESOLVED message should have arrived, and a WARNING process_restarted")
print("   (the API really did restart; it may come a few minutes after the RESOLVED, and its own RESOLVED follows ~5 minutes later).")
print("\n✅ the drill worked: the alert fired while the API was off and cleared when it came back")

0. waiting until the watchdog has seen the API healthy at least once ...
1. stopping the API container (rag) ...
2. 71s after stopping it, the watchdog announced it. Lines from its log:
     FIRING api_unreachable: cannot read /metrics (URLError)"}
     FIRING door_down: the front door answers 502: the API behind it is not answering"}
3. the door itself still answers, honestly: /healthz -> HTTP 502 (502 = the API behind it is down)
   👉 LOOK AT YOUR PHONE: ONE CRITICAL message should have arrived (it names api_unreachable and door_down together).

4. starting the API again ...
   ready after 372s
5. the watchdog announced that it is back.   👉 LOOK AT YOUR PHONE: a RESOLVED message should have arrived, and a WARNING process_restarted
   (the API really did restart; it may come a few minutes after the RESOLVED, and its own RESOLVED follows ~5 minutes later).

✅ the drill worked: the alert fired while the API was off and cleared when it came back


In [18]:
# OPTIONAL: Free public adress; not testes with the real cloudflare

# OPTIONAL. A free Cloudflare "quick tunnel": a public https address that leads to your front door (not to the API).
# For TESTING and sharing with a few people. The address is random and CHANGES whenever the tunnel container restarts.
# Skip this cell if you only want to run locally. (This cell was NOT tested with the real Cloudflare: see the notes.)
assert "DOOR" in globals(), "❌ run Cell 5 first: it defines the helpers this cell uses (and it restarts nothing)"
import tunnel_tools

code, out = dc("compose", "--profile", "tunnel", "up", "-d", "tunnel")
assert code == 0, out
print("tunnel container started; waiting for its address ...")
url = tunnel_tools.wait_for_url(lambda: dc("compose", "logs", "--no-color", "tunnel")[1], 120)
assert url, "❌ no address after 2 minutes: docker compose logs --tail 40 tunnel"
print("address:", url, "   (printed here because you need it; do not share this notebook with its outputs)")
print("PUBLIC_URL", tunnel_tools.set_env_value(".env", "PUBLIC_URL", url), "in .env")
code, out = dc("compose", "up", "-d", "--force-recreate", "--no-deps", "watchdog")      # the watchdog reads PUBLIC_URL when it starts
assert code == 0, out

ok = False
for _ in range(30):                                                          # a new address needs up to a minute before it works everywhere
    try:
        ok = httpx.get(url + "/healthz", timeout=10).status_code == 200
    except Exception:
        ok = False
    if ok:
        break
    time.sleep(3)
print("public /healthz:", "✅ answers" if ok else "❌ not yet (try again in a minute)")
r = subprocess.run([sys.executable, "check_launch.py", "--public", url], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=900)
print(r.stdout.rstrip())

tunnel container started; waiting for its address ...
address: https://violation-wheat-louis-waves.trycloudflare.com    (printed here because you need it; do not share this notebook with its outputs)
PUBLIC_URL unchanged in .env
public /healthz: ✅ answers

1. THE FRONT DOOR  (http://127.0.0.1:8088)
✅ the front door answers /healthz (HTTP 200)
✅ HEAD /healthz works too, for uptime services (HTTP 200)
✅ 52 spellings of closed routes (/metrics, /readyz, /statusz, /docs ...) all answer 404, for GET and for POST
✅ GET /ask is closed (HTTP 404): only POST gets in
✅ the door does not tell strangers what software it is (no Server header)
✅ a question with no key is refused by the API behind the door (HTTP 401): the door reaches the API and the API checks keys
✅ a question with a wrong key is refused (HTTP 401)
✅ a 30 KB question is refused (HTTP 413; 413 = Payload Too Large). The door lets at most 20 KB of any question through to the API

2. THE PUBLIC ADDRESS
✅ the public address answers /hea

After this cell works, run Cell 7 again. With the public address included, the last line can say “READY TO LAUNCH”. If the tunnel container restarts later, its address changes. Then run `python tunnel_tools.py` in a terminal, followed by `docker compose up -d --force-recreate --no-deps watchdog`.

In [20]:
# optional: Who has an APi key?

# OPTIONAL. Who has a key? (names and the last 4 characters only.) To give somebody access, run in a terminal:
#     python manage_keys.py add amina        -> the new key is written to new_key_amina.txt (never printed); give it to her, then delete the file
#     docker compose up -d --force-recreate --no-deps rag
#     (the watchdog cannot tell this planned restart from a crash: expect a CRITICAL message, then RESOLVED a few minutes later, once the models are loaded)
# To take access away:   python manage_keys.py revoke amina   (then the same docker command)
import manage_keys
manage_keys.main(["list"])

  rodney-api           32 characters, ends …BMbd


0

In [21]:
# Daily backuo by windows

# Makes Windows run a backup EVERY DAY, by itself (Task Scheduler). It writes a small file, run_backup.cmd, next to docker-compose.yml and
# registers it. Pick a time when your computer is usually on and Docker Desktop is running. If a backup is ever missed for 2 days, the
# watchdog tells your phone. NOT tested on a real Windows computer: the line "task 'rag-backup' created" below is the proof it worked here.
import backup_tools
if os.name != "nt":
    print("This step is for Windows (Task Scheduler). On Linux or macOS use cron:   0 13 * * *   cd <project folder> && python backup_tools.py backup")
else:
    rep = backup_tools.install_schedule(".", sys.executable, "13:00")
    print(rep.summary())
    assert rep.ok, "❌ read the ❌ line above. Nothing is scheduled. (Remove a half-made task later with:  schtasks /Delete /TN rag-backup /F)"
    print("\nCheck it any time:  schtasks /Query /TN rag-backup      Run it right now to try it:  schtasks /Run /TN rag-backup")
    print("Its log:  backups\\backup_task.log")

✅ wrote run_backup.cmd next to docker-compose.yml (it runs the backup and writes backups\backup_task.log)
✅ task 'rag-backup' created: a backup every day at 13:00  (check: schtasks /Query /TN rag-backup   remove: schtasks /Delete /TN rag-backup /F)
✅ it also runs on battery power, and as soon as possible after a missed start (computer was off)
Docker Desktop must be running at that time. If a backup is missed for 2 days the watchdog tells your phone.

Check it any time:  schtasks /Query /TN rag-backup      Run it right now to try it:  schtasks /Run /TN rag-backup
Its log:  backups\backup_task.log


In [22]:
# when i finish for the day

# Optional, when you finish for the day: stop everything. Your data is kept. (--profile tunnel also stops the tunnel if you started it.)
code, out = dc("compose", "--profile", "tunnel", "down")
print(out or "stopped")
print("Kept: the volumes models, checkpoints, redis_data, caddy_data, caddy_config, and your backups/ folder.")
print("Start again any time with:   docker compose up -d      (and  docker compose --profile tunnel up -d  to include the tunnel)")

Container rag-api-tunnel-1 Stopping 
 Container rag-api-watchdog-1 Stopping 
 Container rag-api-tunnel-1 Stopped 
 Container rag-api-tunnel-1 Removing 
 Container rag-api-watchdog-1 Stopped 
 Container rag-api-watchdog-1 Removing 
 Container rag-api-tunnel-1 Removed 
 Container rag-api-proxy-1 Stopping 
 Container rag-api-watchdog-1 Removed 
 Container rag-api-proxy-1 Stopped 
 Container rag-api-proxy-1 Removing 
 Container rag-api-proxy-1 Removed 
 Container rag-api-rag-1 Stopping 
 Container rag-api-rag-1 Stopped 
 Container rag-api-rag-1 Removing 
 Container rag-api-rag-1 Removed 
 Container rag-api-redis-1 Stopping 
 Container rag-api-redis-1 Stopped 
 Container rag-api-redis-1 Removing 
 Container rag-api-redis-1 Removed 
 Network rag-api_default Removing 
 Network rag-api_default Removed
Kept: the volumes models, checkpoints, redis_data, caddy_data, caddy_config, and your backups/ folder.
Start again any time with:   docker compose up -d      (and  docker compose --profile tunnel